# Stablecoin governance and fragility

**Source-only archival notebook — exploratory research.** Exploratory research design using public-source APIs and author-coded assumptions. Results and model specifications are not independently validated.

All saved outputs and execution counts were removed before public release. No source dataset, independent replication, deployable trading strategy, or verified investment performance is represented here.

Some original data-access paths and dependencies remain unresolved; code may require adaptation to run. Do not interpret results as causal or as a trading signal without additional validation.


# Stablecoin Rule Rigidity and State-Contingent Fragility

This notebook implements a fresh empirical design for the term paper research question:

> In a stablecoin-oriented algorithmic monetary system, what degree of rule rigidity (commitment) maximises long-run welfare under governance capture risk and macroeconomic volatility?

**Empirical strategy.** The notebook demotes the thin cross-section and makes the non-overlapping weekly panel the preferred design. It separates absolute supply fragility, contraction/run-like fragility, and expansion pressure, then estimates whether lagged market stress affects low-, medium-, and high-rigidity designs differently.

**Core claim supported by the design.** The paper should not claim a universal scalar optimum. It should test a conditional optimum: commitment is valuable against discretion and capture, but excessive rigidity may become costly under volatility when adjustment capacity matters.


In [ ]:
# ============================================================
# 0. Imports and global settings
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import os
import time
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

OUTPUT_DIR = "stablecoin_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete.")

## 1. Helper functions

In [ ]:
# ============================================================
# 1. Helper functions
# ============================================================

def safe_get_json(url, params=None, timeout=30, max_retries=3, sleep_seconds=1.0):
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            if response.status_code == 200:
                return response.json()
            last_error = RuntimeError(f"HTTP {response.status_code}: {response.text[:300]}")
            time.sleep(sleep_seconds * attempt)
        except Exception as exc:
            last_error = exc
            time.sleep(sleep_seconds * attempt)
    raise RuntimeError(f"Request failed after {max_retries} attempts: {url}\nLast error: {last_error}")

def get_defillama_stables():
    return safe_get_json("https://stablecoins.llama.fi/stablecoins")

def get_stablecoin_history(stablecoin_id):
    return safe_get_json(f"https://stablecoins.llama.fi/stablecoin/{stablecoin_id}")

def parse_stablecoin_history(symbol, obj):
    rows = []
    def walk(x):
        if isinstance(x, dict):
            if "date" in x or "timestamp" in x:
                rows.append(x)
            for v in x.values():
                walk(v)
        elif isinstance(x, list):
            for item in x:
                walk(item)
    if obj is None:
        return pd.DataFrame()
    walk(obj)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["symbol"] = symbol
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], unit="s", errors="coerce")
    elif "timestamp" in out.columns:
        out["date"] = pd.to_datetime(out["timestamp"], unit="s", errors="coerce")
    else:
        out["date"] = pd.NaT
    return out

def extract_pegged_usd(x):
    if isinstance(x, dict):
        return x.get("peggedUSD", np.nan)
    return np.nan

def extract_bridge_chain_count(x):
    if isinstance(x, dict):
        bridges = x.get("bridges", {})
        if isinstance(bridges, dict):
            return len(bridges)
    return 0

def get_binance_daily_klines(symbol, start_date="2017-01-01", end_date=None):
    from datetime import datetime, timezone
    base_url = "https://api.binance.com/api/v3/klines"
    if end_date is None:
        end_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")
    start_ts = int(pd.Timestamp(start_date, tz="UTC").timestamp() * 1000)
    end_ts = int(pd.Timestamp(end_date, tz="UTC").timestamp() * 1000)
    all_rows = []
    current_start = start_ts
    while current_start < end_ts:
        params = {"symbol": symbol, "interval": "1d", "startTime": current_start, "endTime": end_ts, "limit": 1000}
        response = requests.get(base_url, params=params, timeout=30)
        if response.status_code != 200:
            raise RuntimeError(f"Binance request failed for {symbol}: {response.status_code} | {response.text[:300]}")
        data = response.json()
        if not data:
            break
        all_rows.extend(data)
        current_start = data[-1][0] + 24 * 60 * 60 * 1000
        time.sleep(0.15)
    out = pd.DataFrame(all_rows, columns=["open_time", "open", "high", "low", "close", "volume", "close_time", "quote_asset_volume", "number_of_trades", "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"])
    out["date"] = pd.to_datetime(out["open_time"], unit="ms").dt.normalize()
    out["close"] = pd.to_numeric(out["close"], errors="coerce")
    return out[["date", "close"]].dropna()

def fit_ols(formula, data, cov_type="HC3", cov_kwds=None):
    if cov_kwds is None:
        cov_kwds = {}
    return smf.ols(formula, data=data).fit(cov_type=cov_type, cov_kwds=cov_kwds)

print("Helper functions loaded.")

## 2. Stablecoin sample and governance coding

Edit this coding table if you refine the classification. The main models use `rigidity_bin` rather than relying on false precision in a 0–5 index.

In [ ]:
# ============================================================
# 2. Stablecoin sample and governance coding
# ============================================================

expanded_symbols = [
    "USDT", "USDC", "DAI", "FRAX", "LUSD", "USDP", "GUSD",
    "TUSD", "BUSD", "USDD", "SUSD", "MIM", "USDE", "FDUSD",
    "PYUSD", "USDL", "USD0", "USDM", "DOLA", "RAI"
]

governance_scores = pd.DataFrame([
    ["USDT", "fiat_custodial", 1.5, 4.5, 5.0, 1.0, 0],
    ["USDC", "fiat_custodial", 2.0, 3.5, 5.0, 1.0, 0],
    ["DAI", "crypto_collateralized", 3.0, 2.5, 3.0, 5.0, 0],
    ["FRAX", "hybrid_algorithmic", 2.5, 3.0, 4.0, 4.0, 1],
    ["LUSD", "immutable_crypto_collateralized", 4.5, 1.5, 1.5, 4.0, 0],
    ["USDP", "fiat_custodial", 2.0, 3.5, 5.0, 1.0, 0],
    ["GUSD", "fiat_custodial", 2.0, 3.5, 5.0, 1.0, 0],
    ["TUSD", "fiat_custodial", 2.0, 4.0, 5.0, 1.0, 0],
    ["BUSD", "fiat_custodial", 2.0, 4.0, 5.0, 1.0, 0],
    ["FDUSD", "fiat_custodial", 2.0, 4.0, 5.0, 1.0, 0],
    ["PYUSD", "fiat_custodial", 2.0, 3.5, 5.0, 1.0, 0],
    ["USDD", "algorithmic_hybrid", 3.5, 3.5, 3.5, 3.0, 1],
    ["MIM", "crypto_collateralized", 3.0, 3.0, 3.0, 4.0, 0],
    ["SUSD", "synthetic_crypto_backed", 2.5, 3.0, 4.0, 5.0, 0],
    ["USDE", "synthetic_delta_neutral", 2.5, 3.5, 4.0, 4.0, 0],
    ["DOLA", "crypto_credit_stablecoin", 2.5, 3.0, 3.5, 4.0, 0],
    ["RAI", "reflex_index", 4.0, 2.0, 2.0, 5.0, 0],
    ["USD0", "rwa_reserve", 2.5, 3.0, 4.0, 2.0, 0],
    ["USDM", "rwa_reserve", 2.5, 3.0, 4.0, 2.0, 0],
    ["USDL", "rwa_reserve", 2.5, 3.0, 4.0, 2.0, 0],
], columns=["symbol", "stablecoin_type", "rigidity_score", "capture_risk_score", "emergency_discretion_score", "oracle_dependence_score", "collateral_endogeneity"])

governance_scores["symbol"] = governance_scores["symbol"].str.upper().str.strip()
governance_scores

## 3. Download and parse DeFiLlama data

In [ ]:
# ============================================================
# 3. Download and parse DeFiLlama stablecoin data
# ============================================================

stables_raw = get_defillama_stables()
stables = pd.DataFrame(stables_raw["peggedAssets"])
stables["symbol_clean"] = stables["symbol"].astype(str).str.upper().str.strip()
selected = stables[stables["symbol_clean"].isin(expanded_symbols)].copy()

display_cols = [c for c in ["id", "name", "symbol", "pegType", "pegMechanism", "priceSource"] if c in selected.columns]
print("Selected stablecoins from DeFiLlama:")
display(selected[display_cols].sort_values("symbol"))

histories = {}
for _, row in selected.iterrows():
    sym = row["symbol_clean"]
    sid = row["id"]
    print(f"Downloading {sym} | {row.get('name', '')} | ID {sid}")
    try:
        histories[sym] = get_stablecoin_history(sid)
    except Exception as exc:
        print(f"  Failed {sym}: {exc}")
        histories[sym] = None
    time.sleep(0.5)

parsed = []
for symbol, obj in histories.items():
    temp = parse_stablecoin_history(symbol, obj)
    if not temp.empty:
        parsed.append(temp)

if not parsed:
    raise RuntimeError("No stablecoin histories parsed. Check DeFiLlama connectivity or selected symbols.")

raw = pd.concat(parsed, ignore_index=True)
print("Raw parsed shape:", raw.shape)
print(raw["symbol"].value_counts().head(30))
display(raw.head())

## 4. Construct stablecoin-day panel

In [ ]:
# ============================================================
# 4. Construct stablecoin-day panel
# ============================================================

df = raw.copy()
df["circulating_usd"] = df["circulating"].apply(extract_pegged_usd)
df["bridged_usd"] = df["bridgedTo"].apply(extract_pegged_usd) if "bridgedTo" in df.columns else np.nan
df["bridge_chain_count"] = df["bridgedTo"].apply(extract_bridge_chain_count) if "bridgedTo" in df.columns else 0

df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df["symbol"] = df["symbol"].astype(str).str.upper().str.strip()
df = df.merge(governance_scores, on="symbol", how="left")
df = df[df["rigidity_score"].notna()].copy()

df["bridged_usd"] = df["bridged_usd"].fillna(0)
df["bridge_chain_count"] = df["bridge_chain_count"].fillna(0)
df["bridged_share"] = (df["bridged_usd"] / df["circulating_usd"]).replace([np.inf, -np.inf], np.nan)
df["bridged_share_w"] = df["bridged_share"].clip(upper=df["bridged_share"].quantile(0.99)).fillna(0)
df["log_bridge_chain_count"] = np.log1p(df["bridge_chain_count"])

df["terra_window"] = ((df["date"] >= "2022-05-07") & (df["date"] <= "2022-05-20")).astype(int)
df["ftx_window"] = ((df["date"] >= "2022-11-06") & (df["date"] <= "2022-11-20")).astype(int)
df["svb_window"] = ((df["date"] >= "2023-03-10") & (df["date"] <= "2023-03-20")).astype(int)
df["svb_treated"] = df["symbol"].isin(["USDC", "DAI", "FRAX"]).astype(int)
df["svb_treated_post"] = df["svb_treated"] * df["svb_window"]

daily = df.groupby(["symbol", "date"], as_index=False).agg(
    circulating_usd=("circulating_usd", "max"),
    bridged_usd=("bridged_usd", "max"),
    bridged_share_w=("bridged_share_w", "max"),
    bridge_chain_count=("bridge_chain_count", "max"),
    log_bridge_chain_count=("log_bridge_chain_count", "max"),
    terra_window=("terra_window", "max"),
    ftx_window=("ftx_window", "max"),
    svb_window=("svb_window", "max"),
    svb_treated=("svb_treated", "max"),
    svb_treated_post=("svb_treated_post", "max"),
    stablecoin_type=("stablecoin_type", "first"),
    rigidity_score=("rigidity_score", "first"),
    capture_risk_score=("capture_risk_score", "first"),
    emergency_discretion_score=("emergency_discretion_score", "first"),
    oracle_dependence_score=("oracle_dependence_score", "first"),
    collateral_endogeneity=("collateral_endogeneity", "first")
)

daily = daily.sort_values(["symbol", "date"])
daily = daily[daily["circulating_usd"] >= 1_000_000].copy()
daily["first_date"] = daily.groupby("symbol")["date"].transform("min")
daily["coin_age_days"] = (daily["date"] - daily["first_date"]).dt.days
daily = daily[daily["coin_age_days"] >= 180].copy()

print("Daily panel shape:", daily.shape)
print(daily.groupby("symbol")["date"].count().sort_values(ascending=False))
display(daily.head())

## 5. Market stress proxy from BTC/ETH volatility

In [ ]:
# ============================================================
# 5. Market stress from Binance BTC/ETH volatility
# ============================================================

try:
    btc = get_binance_daily_klines("BTCUSDT", start_date="2017-01-01").rename(columns={"close": "btc_price"})
    eth = get_binance_daily_klines("ETHUSDT", start_date="2017-01-01").rename(columns={"close": "eth_price"})
    btc["btc_return"] = btc["btc_price"].pct_change()
    eth["eth_return"] = eth["eth_price"].pct_change()
    btc["btc_volatility_30d"] = btc["btc_return"].rolling(30).std()
    eth["eth_volatility_30d"] = eth["eth_return"].rolling(30).std()
    stress = btc[["date", "btc_volatility_30d"]].merge(eth[["date", "eth_volatility_30d"]], on="date", how="outer")
    stress["crypto_market_stress"] = stress[["btc_volatility_30d", "eth_volatility_30d"]].mean(axis=1)
    stress["market_stress_z"] = (stress["crypto_market_stress"] - stress["crypto_market_stress"].mean()) / stress["crypto_market_stress"].std()
    stress = stress[["date", "market_stress_z"]].dropna()
    stress["date"] = pd.to_datetime(stress["date"]).dt.normalize()
except Exception as exc:
    print("Market stress API failed. Falling back to event-based stress proxy.")
    print(exc)
    stress = daily[["date"]].drop_duplicates().copy()
    stress["terra_window"] = ((stress["date"] >= "2022-05-07") & (stress["date"] <= "2022-05-20")).astype(int)
    stress["ftx_window"] = ((stress["date"] >= "2022-11-06") & (stress["date"] <= "2022-11-20")).astype(int)
    stress["svb_window"] = ((stress["date"] >= "2023-03-10") & (stress["date"] <= "2023-03-20")).astype(int)
    stress["market_stress_z"] = 2.0 * stress["terra_window"] + 1.5 * stress["ftx_window"] + 1.5 * stress["svb_window"]
    stress = stress[["date", "market_stress_z"]]

daily = daily.drop(columns=["market_stress_z"], errors="ignore")
daily = daily.merge(stress, on="date", how="left")
daily["market_stress_z"] = daily["market_stress_z"].fillna(0)

print(stress.head())
print(stress.tail())
print(daily["market_stress_z"].describe())

## 6. Preferred non-overlapping weekly panel

In [ ]:
# ============================================================
# 6. Build non-overlapping weekly panel
# ============================================================

weekly_base = daily.copy()
weekly_base["week"] = weekly_base["date"].dt.to_period("W").apply(lambda r: r.start_time)

weekly = weekly_base.groupby(["symbol", "week"], as_index=False).agg(
    circulating_usd=("circulating_usd", "last"),
    bridged_share_w=("bridged_share_w", "mean"),
    bridge_chain_count=("bridge_chain_count", "max"),
    market_stress_z=("market_stress_z", "mean"),
    terra_window=("terra_window", "max"),
    ftx_window=("ftx_window", "max"),
    svb_window=("svb_window", "max"),
    svb_treated=("svb_treated", "max"),
    svb_treated_post=("svb_treated_post", "max"),
    stablecoin_type=("stablecoin_type", "first"),
    rigidity_score=("rigidity_score", "first"),
    capture_risk_score=("capture_risk_score", "first"),
    emergency_discretion_score=("emergency_discretion_score", "first"),
    oracle_dependence_score=("oracle_dependence_score", "first"),
    collateral_endogeneity=("collateral_endogeneity", "first")
)

weekly = weekly.sort_values(["symbol", "week"])
weekly["weekly_supply_growth"] = weekly.groupby("symbol")["circulating_usd"].pct_change()
weekly["absolute_fragility"] = weekly["weekly_supply_growth"].abs()
weekly["contraction_fragility"] = np.maximum(-weekly["weekly_supply_growth"], 0)
weekly["expansion_pressure"] = np.maximum(weekly["weekly_supply_growth"], 0)

for col in ["absolute_fragility", "contraction_fragility", "expansion_pressure"]:
    upper = weekly[col].quantile(0.99)
    weekly[col + "_w"] = weekly[col].clip(upper=upper)
    weekly["log_" + col] = np.log1p(weekly[col + "_w"])

weekly["log_circulating_usd"] = np.log1p(weekly["circulating_usd"])
weekly["log_bridge_chain_count"] = np.log1p(weekly["bridge_chain_count"].fillna(0))
weekly["market_stress_lag"] = weekly.groupby("symbol")["market_stress_z"].shift(1)
weekly["bridged_share_lag"] = weekly.groupby("symbol")["bridged_share_w"].shift(1)
weekly["log_circulating_lag"] = weekly.groupby("symbol")["log_circulating_usd"].shift(1)

weekly["rigidity_bin"] = pd.cut(weekly["rigidity_score"], bins=[0, 2.0, 3.5, 5.0], labels=["low", "medium", "high"], include_lowest=True)
weekly["design_bin"] = np.select(
    [
        weekly["stablecoin_type"].str.contains("fiat|rwa", case=False, na=False),
        weekly["stablecoin_type"].str.contains("crypto|synthetic|credit|reflex", case=False, na=False),
        weekly["stablecoin_type"].str.contains("algorithmic|hybrid", case=False, na=False),
    ],
    ["reserve_or_custodial", "crypto_or_synthetic", "algorithmic_or_hybrid"],
    default="other"
)
weekly["month"] = weekly["week"].dt.to_period("M").astype(str)

weekly_reg = weekly.dropna(subset=["log_absolute_fragility", "log_contraction_fragility", "log_expansion_pressure", "market_stress_lag", "bridged_share_lag", "log_circulating_lag", "rigidity_bin", "symbol", "month"]).copy()
valid_symbols = weekly_reg.groupby("symbol")["week"].count()
valid_symbols = valid_symbols[valid_symbols >= 26].index
weekly_reg = weekly_reg[weekly_reg["symbol"].isin(valid_symbols)].copy()

print("Weekly regression sample:", weekly_reg.shape)
print(weekly_reg["symbol"].value_counts().sort_values(ascending=False))
display(weekly_reg.head())

## 7. Main weekly models

In [ ]:
# ============================================================
# 7. Main weekly models
# ============================================================

baseline_formula = """
{outcome} ~ market_stress_lag
          + C(rigidity_bin):market_stress_lag
          + C(design_bin):market_stress_lag
          + bridged_share_lag
          + log_circulating_lag
          + terra_window
          + ftx_window
          + svb_treated_post
          + C(symbol)
          + C(month)
"""

model_weekly_absolute = fit_ols(baseline_formula.format(outcome="log_absolute_fragility"), weekly_reg, cov_type="HC3")
model_weekly_contraction = fit_ols(baseline_formula.format(outcome="log_contraction_fragility"), weekly_reg, cov_type="HC3")
model_weekly_expansion = fit_ols(baseline_formula.format(outcome="log_expansion_pressure"), weekly_reg, cov_type="HC3")

print("=== Absolute Fragility ===")
print(model_weekly_absolute.summary().tables[1])
print("\n=== Contraction Fragility ===")
print(model_weekly_contraction.summary().tables[1])
print("\n=== Expansion Pressure ===")
print(model_weekly_expansion.summary().tables[1])

## 8. Compact results table

In [ ]:
# ============================================================
# 8. Compact coefficient table
# ============================================================

def extract_terms(model, model_name, terms=None):
    if terms is None:
        terms = [
            "market_stress_lag",
            "C(rigidity_bin)[low]:market_stress_lag",
            "C(rigidity_bin)[medium]:market_stress_lag",
            "C(rigidity_bin)[high]:market_stress_lag",
            "bridged_share_lag",
            "log_circulating_lag",
            "terra_window",
            "ftx_window",
            "svb_treated_post"
        ]
    rows = []
    ci = model.conf_int()
    for term in terms:
        if term in model.params.index:
            rows.append({"model": model_name, "term": term, "coef": model.params[term], "se": model.bse[term], "pvalue": model.pvalues[term], "ci_low": ci.loc[term, 0], "ci_high": ci.loc[term, 1]})
    return pd.DataFrame(rows)

coef_table = pd.concat([
    extract_terms(model_weekly_absolute, "Absolute fragility"),
    extract_terms(model_weekly_contraction, "Contraction fragility"),
    extract_terms(model_weekly_expansion, "Expansion pressure"),
], ignore_index=True)

coef_table["stars"] = pd.cut(coef_table["pvalue"], bins=[-np.inf, 0.01, 0.05, 0.10, np.inf], labels=["***", "**", "*", ""])
display(coef_table)
coef_table.to_csv(f"{OUTPUT_DIR}/main_weekly_results.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/main_weekly_results.csv")

## 9. Descriptive tables and figures

In [ ]:
# ============================================================
# 9. Descriptive tables and figures
# ============================================================

coin_summary = weekly_reg.groupby("symbol").agg(
    stablecoin_type=("stablecoin_type", "first"),
    design_bin=("design_bin", "first"),
    rigidity_score=("rigidity_score", "first"),
    rigidity_bin=("rigidity_bin", "first"),
    capture_risk_score=("capture_risk_score", "first"),
    oracle_dependence_score=("oracle_dependence_score", "first"),
    collateral_endogeneity=("collateral_endogeneity", "first"),
    observations=("week", "count"),
    mean_circulating_usd=("circulating_usd", "mean"),
    mean_absolute_fragility=("log_absolute_fragility", "mean"),
    mean_contraction_fragility=("log_contraction_fragility", "mean"),
    mean_expansion_pressure=("log_expansion_pressure", "mean"),
    mean_market_stress=("market_stress_lag", "mean"),
    mean_bridged_share=("bridged_share_lag", "mean")
).reset_index().sort_values("mean_absolute_fragility", ascending=False)

display(coin_summary)
coin_summary.to_csv(f"{OUTPUT_DIR}/coin_summary.csv", index=False)

fig_df = coin_summary.sort_values("mean_absolute_fragility", ascending=True)
plt.figure(figsize=(9, 6))
plt.barh(fig_df["symbol"], fig_df["mean_absolute_fragility"])
plt.xlabel("Mean log absolute weekly supply fragility")
plt.ylabel("Stablecoin")
plt.title("Average Supply Fragility by Stablecoin")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_mean_fragility_by_coin.png", dpi=200)
plt.show()

plot_df = coin_summary.dropna(subset=["rigidity_score", "mean_absolute_fragility"]).copy()
plt.figure(figsize=(8, 6))
plt.scatter(plot_df["rigidity_score"], plot_df["mean_absolute_fragility"], s=(plot_df["capture_risk_score"] + 1) * 70)
for _, row in plot_df.iterrows():
    plt.text(row["rigidity_score"] + 0.03, row["mean_absolute_fragility"], row["symbol"], fontsize=8)
plt.xlabel("Rule rigidity score")
plt.ylabel("Mean log absolute weekly supply fragility")
plt.title("Rule Rigidity and Stablecoin Supply Fragility")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_rigidity_fragility_scatter.png", dpi=200)
plt.show()

## 10. SVB dynamic event study

In [ ]:

# ============================================================
# 10. SVB dynamic event study
# ============================================================

event_df = weekly_reg.copy()

svb_date = pd.Timestamp("2023-03-10")
svb_week = svb_date.to_period("W").start_time

event_df["svb_relative_week"] = (
    (event_df["week"] - svb_week).dt.days // 7
).astype(int)

# Keep a symmetric event window
svb_es = event_df[
    (event_df["svb_relative_week"] >= -8) &
    (event_df["svb_relative_week"] <= 8)
].copy()

# Treated group: stablecoins plausibly exposed to the USDC/SVB channel
# This should already exist, but this line makes the cell self-contained.
svb_es["svb_treated"] = svb_es["symbol"].isin(["USDC", "DAI", "FRAX"]).astype(int)

# Create Patsy-safe dummy names:
# k = -8 becomes svb_m8
# k =  0 becomes svb_0
# k =  8 becomes svb_p8
event_terms_list = []

for k in range(-8, 9):
    if k == -1:
        # omit week -1 as the baseline
        continue

    if k < 0:
        col = f"svb_m{abs(k)}"
    elif k > 0:
        col = f"svb_p{k}"
    else:
        col = "svb_0"

    svb_es[col] = (
        (svb_es["svb_relative_week"] == k) &
        (svb_es["svb_treated"] == 1)
    ).astype(int)

    event_terms_list.append(col)

event_terms = " + ".join(event_terms_list)

if len(svb_es) > 0 and event_terms:
    svb_formula = f"""
    log_absolute_fragility ~ {event_terms}
                           + C(symbol)
                           + C(week)
    """

    model_svb_event = fit_ols(
        svb_formula,
        svb_es,
        cov_type="HC3"
    )

    print(model_svb_event.summary().tables[1])

    es_rows = []
    ci = model_svb_event.conf_int()

    for k in range(-8, 9):
        if k == -1:
            continue

        if k < 0:
            term = f"svb_m{abs(k)}"
        elif k > 0:
            term = f"svb_p{k}"
        else:
            term = "svb_0"

        if term in model_svb_event.params.index:
            es_rows.append({
                "relative_week": k,
                "term": term,
                "coef": model_svb_event.params[term],
                "se": model_svb_event.bse[term],
                "pvalue": model_svb_event.pvalues[term],
                "ci_low": ci.loc[term, 0],
                "ci_high": ci.loc[term, 1],
            })

    svb_event_table = pd.DataFrame(es_rows)
    display(svb_event_table)

    svb_event_table.to_csv(
        f"{OUTPUT_DIR}/svb_event_study.csv",
        index=False
    )

    plt.figure(figsize=(8, 5))
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linestyle="--", linewidth=1)

    plt.errorbar(
        svb_event_table["relative_week"],
        svb_event_table["coef"],
        yerr=1.96 * svb_event_table["se"],
        fmt="o"
    )

    plt.xlabel("Weeks relative to SVB shock")
    plt.ylabel("Treated × event-week coefficient")
    plt.title("SVB Event Study: Exposed Stablecoins")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/fig_svb_event_study.png", dpi=200)
    plt.show()

else:
    print("Insufficient SVB event data.")

## 11. Common-shock local projections: Terra and FTX

In [ ]:
# ============================================================
# 11. Simple local projections for common shocks
# ============================================================

def common_shock_lp(data, shock_date, outcome="log_absolute_fragility", max_horizon=8, shock_name="shock"):
    temp = data.copy()
    shock_week = pd.Timestamp(shock_date).to_period("W").start_time
    agg = temp.groupby("week", as_index=False).agg(y=(outcome, "mean"))
    agg = agg.sort_values("week")
    agg["shock"] = (agg["week"] == shock_week).astype(int)
    rows = []
    for h in range(0, max_horizon + 1):
        lp = agg.copy()
        lp[f"y_lead_{h}"] = lp["y"].shift(-h)
        lp = lp.dropna(subset=[f"y_lead_{h}", "shock"]).copy()
        if lp["shock"].sum() == 0:
            continue
        m = fit_ols(f"y_lead_{h} ~ shock", lp, cov_type="HC3")
        rows.append({"shock_name": shock_name, "horizon": h, "coef": m.params.get("shock", np.nan), "se": m.bse.get("shock", np.nan), "pvalue": m.pvalues.get("shock", np.nan)})
    return pd.DataFrame(rows)

terra_lp = common_shock_lp(weekly_reg, "2022-05-07", shock_name="Terra")
ftx_lp = common_shock_lp(weekly_reg, "2022-11-06", shock_name="FTX")
lp_table = pd.concat([terra_lp, ftx_lp], ignore_index=True)
display(lp_table)
lp_table.to_csv(f"{OUTPUT_DIR}/common_shock_local_projections.csv", index=False)

plt.figure(figsize=(8, 5))
for name, grp in lp_table.groupby("shock_name"):
    plt.errorbar(grp["horizon"], grp["coef"], yerr=1.96 * grp["se"], marker="o", label=name)
plt.axhline(0, linewidth=1)
plt.xlabel("Weeks after shock")
plt.ylabel("Response of average fragility")
plt.title("Dynamic Response of Average Stablecoin Fragility")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/fig_common_shock_lp.png", dpi=200)
plt.show()

## 12. Leave-one-out robustness

In [ ]:
# ============================================================
# 12. Leave-one-out robustness
# ============================================================

drop_list = ["USDT", "USDC", "DAI", "FRAX", "LUSD"]
robust_rows = []
for coin in drop_list:
    temp = weekly_reg[weekly_reg["symbol"] != coin].copy()
    try:
        m = fit_ols(baseline_formula.format(outcome="log_absolute_fragility"), temp, cov_type="HC3")
        for term in ["market_stress_lag", "C(rigidity_bin)[medium]:market_stress_lag", "C(rigidity_bin)[high]:market_stress_lag", "terra_window", "svb_treated_post"]:
            if term in m.params.index:
                robust_rows.append({"dropped_coin": coin, "term": term, "coef": m.params[term], "se": m.bse[term], "pvalue": m.pvalues[term]})
    except Exception as exc:
        robust_rows.append({"dropped_coin": coin, "term": "MODEL_FAILED", "coef": np.nan, "se": np.nan, "pvalue": np.nan, "error": str(exc)})
robust_table = pd.DataFrame(robust_rows)
display(robust_table)
robust_table.to_csv(f"{OUTPUT_DIR}/leave_one_out_robustness.csv", index=False)

## 13. Appendix-only daily overlapping model

This is not the preferred model because the dependent variable uses overlapping 7-day windows. Keep it as appendix/robustness only.

In [ ]:
# ============================================================
# 13. Daily overlapping model for appendix only
# ============================================================

daily_app = daily.copy().sort_values(["symbol", "date"])
daily_app["supply_growth_7d"] = daily_app.groupby("symbol")["circulating_usd"].pct_change(7)
daily_app["abs_supply_growth_7d"] = daily_app["supply_growth_7d"].abs()
daily_app["abs_supply_growth_7d_w"] = daily_app["abs_supply_growth_7d"].clip(upper=daily_app["abs_supply_growth_7d"].quantile(0.99))
daily_app["log_supply_fragility"] = np.log1p(daily_app["abs_supply_growth_7d_w"])
daily_app["log_circulating_usd"] = np.log1p(daily_app["circulating_usd"])
daily_app["rigidity_score_centered"] = daily_app["rigidity_score"] - daily_app["rigidity_score"].mean()
daily_app["year_month"] = daily_app["date"].dt.to_period("M").astype(str)

daily_reg = daily_app.dropna(subset=["log_supply_fragility", "market_stress_z", "rigidity_score_centered", "bridged_share_w", "log_circulating_usd", "symbol", "year_month"]).copy()

daily_model = fit_ols("""
log_supply_fragility ~ market_stress_z
                     + rigidity_score_centered:market_stress_z
                     + bridged_share_w
                     + log_circulating_usd
                     + terra_window
                     + ftx_window
                     + svb_treated_post
                     + C(symbol)
                     + C(year_month)
""", daily_reg, cov_type="HC3")

print(daily_model.summary())
extract_terms(daily_model, "Daily overlapping appendix", terms=["market_stress_z", "rigidity_score_centered:market_stress_z", "bridged_share_w", "log_circulating_usd", "terra_window", "ftx_window", "svb_treated_post"]).to_csv(f"{OUTPUT_DIR}/daily_appendix_results.csv", index=False)

## 14. Automatic interpretation aid

In [ ]:
# ============================================================
# 14. Automatic interpretation aid
# ============================================================

def interpret_coef(model, term, plain_name):
    if term not in model.params.index:
        return f"{plain_name}: term not present in model."
    coef = model.params[term]
    p = model.pvalues[term]
    if p < 0.01:
        sig = "statistically significant at the 1% level"
    elif p < 0.05:
        sig = "statistically significant at the 5% level"
    elif p < 0.10:
        sig = "marginally significant at the 10% level"
    else:
        sig = "not statistically significant"
    direction = "positive" if coef > 0 else "negative"
    return f"{plain_name}: coefficient = {coef:.4f}, p = {p:.3f}; {direction} and {sig}."

print("Preferred weekly absolute-fragility model:")
for term, name in [
    ("market_stress_lag", "Lagged market stress"),
    ("C(rigidity_bin)[medium]:market_stress_lag", "Medium rigidity × stress"),
    ("C(rigidity_bin)[high]:market_stress_lag", "High rigidity × stress"),
    ("bridged_share_lag", "Lagged bridged share"),
    ("log_circulating_lag", "Lagged size"),
    ("terra_window", "Terra window"),
    ("ftx_window", "FTX window"),
    ("svb_treated_post", "SVB treated × post"),
]:
    print("-", interpret_coef(model_weekly_absolute, term, name))

print("\nPaper-safe bottom line:")
print("The preferred weekly panel should be interpreted as evidence on state-contingent fragility, not as a universal scalar estimate of optimal rigidity.")

## 15. Save final analytical datasets

In [ ]:
# ============================================================
# 15. Save analytical datasets
# ============================================================

daily.to_csv(f"{OUTPUT_DIR}/daily_panel_clean.csv", index=False)
weekly_reg.to_csv(f"{OUTPUT_DIR}/weekly_panel_regression.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/daily_panel_clean.csv")
print(f"Saved: {OUTPUT_DIR}/weekly_panel_regression.csv")


# Final paper claim supported by this notebook

A conservative, defensible results statement is:

> The empirical analysis does not support a single welfare-maximising rigidity score that applies across stablecoins. Instead, it supports a conditional optimum. In non-overlapping weekly panel models, the relevant empirical question is whether market stress affects low-, medium-, and high-rigidity systems differently. This specification is better aligned with the data than a thin 17-coin cross-section because stablecoin fixed effects absorb time-invariant governance design. The paper therefore interprets rule rigidity as a state-contingent design choice: commitment is valuable against discretion and capture, but excessive rigidity can become costly when volatility requires adjustment.


In [ ]:

model_weekly_absolute_cluster = smf.ols(
    """
    log_absolute_fragility ~ market_stress_lag
                           + C(rigidity_bin):market_stress_lag
                           + C(design_bin):market_stress_lag
                           + bridged_share_lag
                           + log_circulating_lag
                           + terra_window
                           + ftx_window
                           + svb_treated_post
                           + C(symbol)
                           + C(month)
    """,
    data=weekly_reg
).fit(
    cov_type="cluster",
    cov_kwds={"groups": weekly_reg["symbol"]}
)

print(model_weekly_absolute_cluster.summary())

In [ ]:
import statsmodels.formula.api as smf

quantile_formula = """
log_absolute_fragility ~ market_stress_lag
                       + C(rigidity_bin):market_stress_lag
                       + bridged_share_lag
                       + log_circulating_lag
                       + terra_window
                       + ftx_window
                       + svb_treated_post
                       + C(symbol)
                       + C(month)
"""

q50 = smf.quantreg(quantile_formula, weekly_reg).fit(q=0.50)
q90 = smf.quantreg(quantile_formula, weekly_reg).fit(q=0.90)

print("Median / q=0.50")
print(q50.summary())

print("Upper-tail / q=0.90")
print(q90.summary())

In [ ]:

run_df = weekly_reg.copy()

# Define run-like contraction threshold
run_threshold = 0.10

run_df["run_event"] = (
    run_df["weekly_supply_growth"] <= -run_threshold
).astype(int)

print(run_df["run_event"].value_counts(normalize=True))

logit_model = smf.logit(
    """
    run_event ~ market_stress_lag
              + C(rigidity_bin):market_stress_lag
              + bridged_share_lag
              + log_circulating_lag
              + terra_window
              + ftx_window
              + svb_treated_post
              + C(symbol)
              + C(month)
    """,
    data=run_df
).fit(maxiter=500)

print(logit_model.summary())

In [ ]:
# ============================================================
# Run-event descriptive table
# ============================================================

run_summary = (
    run_df.groupby("symbol")
    .agg(
        observations=("run_event", "count"),
        run_events=("run_event", "sum"),
        run_rate=("run_event", "mean"),
        rigidity_score=("rigidity_score", "first"),
        rigidity_bin=("rigidity_bin", "first"),
        stablecoin_type=("stablecoin_type", "first"),
        mean_market_stress=("market_stress_lag", "mean"),
        mean_size=("log_circulating_lag", "mean"),
        mean_bridge=("bridged_share_lag", "mean")
    )
    .reset_index()
    .sort_values("run_rate", ascending=False)
)

display(run_summary)

In [ ]:
# ============================================================
# Binary run-risk robustness: Linear Probability Model
# ============================================================

run_lpm = smf.ols(
    """
    run_event ~ market_stress_lag
              + C(rigidity_bin):market_stress_lag
              + bridged_share_lag
              + log_circulating_lag
              + terra_window
              + ftx_window
              + svb_treated_post
              + C(symbol)
              + C(month)
    """,
    data=run_df
).fit(cov_type="HC3")

print(run_lpm.summary())

In [ ]:
# ============================================================
# Regularised logistic model: simplified specification
# ============================================================

logit_reg = smf.logit(
    """
    run_event ~ market_stress_lag
              + C(rigidity_bin):market_stress_lag
              + bridged_share_lag
              + log_circulating_lag
              + terra_window
              + ftx_window
              + svb_treated_post
              + C(symbol)
    """,
    data=run_df
).fit_regularized(alpha=0.1, maxiter=1000)

print(logit_reg.params)

In [ ]:
logit_reg_stronger = smf.logit(
    """
    run_event ~ market_stress_lag
              + C(rigidity_bin):market_stress_lag
              + bridged_share_lag
              + log_circulating_lag
              + terra_window
              + ftx_window
              + svb_treated_post
              + C(symbol)
    """,
    data=run_df
).fit_regularized(alpha=1.0, maxiter=1000)

print(logit_reg_stronger.params)

In [ ]:
# ============================================================
# Parsimonious logit model for rare run events
# ============================================================

run_logit_simple = smf.logit(
    """
    run_event ~ market_stress_lag
              + C(rigidity_bin):market_stress_lag
              + bridged_share_lag
              + log_circulating_lag
              + terra_window
              + ftx_window
              + svb_treated_post
    """,
    data=run_df
).fit(maxiter=1000)

print(run_logit_simple.summary())